In [1]:
!pip install pandas scikit-learn spacy -q


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import json
import sys
import os

In [3]:
if 'google.colab' in sys.modules: 
    if not os.path.exists('/content/nlp_uni'):
        !git clone -b lab-04 https://github.com/Danylo-NULP/nlp_uni.git
    
    %cd /content/nlp_uni
    !pip install pandas scikit-learn spacy -q
    sys.path.append('/content/nlp_uni')
    
    FOLDER_ID = '1LhS2rA8VAQVd_lzUwMXuHav6fSVcGO0D'
    
    os.makedirs('/content/nlp_uni/data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O /content/nlp_uni/data/
    
    data_dir = '/content/nlp_uni/data'

else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data'

In [4]:
import json
import os
import sys
import pandas as pd
from IPython.display import display

# Підтягуємо нашу функцію з модуля
from src.ie_rules import extract_all

print("=== 4. Тестування Edge Cases ===\n")

# Визначаємо шлях до тестів
if 'google.colab' in sys.modules: 
    edge_cases_path = '/content/nlp_uni/tests/ie_edge_cases.jsonl'
else:
    edge_cases_path = '../tests/ie_edge_cases.jsonl'

results_data = []

# Читаємо jsonl та проганяємо через екстрактор
with open(edge_cases_path, 'r', encoding='utf-8') as f:
    for line in f:
        case = json.loads(line)
        text = case['raw_text']
        expected = case['expected_behavior']
        
        # ВИКЛИК НАШОЇ ФУНКЦІЇ
        extracted_dict = extract_all(text)
        
        # Форматуємо результат для красивого виводу
        extracted_list = []
        for f_type, items in extracted_dict.items():
            for item in items:
                extracted_list.append(f"{item['value']} [{f_type}]")
                
        extracted_str = ", ".join(extracted_list) if extracted_list else "Нічого"

        results_data.append({
            "Текст (raw_text)": text,
            "Знайдені сутності": extracted_str,
            "Очікувана поведінка": expected
        })

df_edges = pd.DataFrame(results_data)

pd.set_option('display.max_colwidth', None)
display(df_edges.head(15))

=== 4. Тестування Edge Cases ===



,Текст (raw_text),Знайдені сутності,Очікувана поведінка
0,A one-way street.,street [LOCATION],не витягувати 'one' з дефісом
1,Someone is walking.,Нічого,не витягувати 'one' зі слова 'Someone' (працює завдяки \b)
2,Catch-22 is a great book.,Нічого,не вважати 22 кількістю об'єктів на фото
3,He is 100% sure.,Нічого,не витягувати відсотки як кількість
4,A group of friends.,many [QUANTITY],слово 'group' нормалізується до 'many'
5,Player number 3 is running.,Нічого,не витягувати номер гравця як кількість
6,He has a black eye.,Нічого,не вважати 'black eye' (синець) кольором одягу
7,Officer of the U.S. Navy.,Нічого,не витягувати 'Navy' як 'blue' (це організація)
8,It happened out of the blue.,Нічого,не витягувати 'blue' з ідіоми
9,The Red Cross tent.,Нічого,не витягувати 'Red' з власної назви


In [5]:
import pandas as pd
import os
from src.ie_rules import extract_all

print("5. Оцінка на Gold Subset (Dynamic Precision)\n")

# 1. Задаємо еталон (Gold Subset) для SNLI
gold_data = [
    {"text_id": 1, "text": "Two men are walking.", "field_type": "QUANTITY", "normalized_value": "2"},
    {"text_id": 2, "text": "A group of people sitting.", "field_type": "QUANTITY", "normalized_value": "many"},
    {"text_id": 3, "text": "3 dogs playing.", "field_type": "QUANTITY", "normalized_value": "3"},
    {"text_id": 4, "text": "Five birds in the sky.", "field_type": "QUANTITY", "normalized_value": "5"},
    {"text_id": 5, "text": "Man in a red shirt.", "field_type": "COLOR", "normalized_value": "red"},
    {"text_id": 6, "text": "A black dog running.", "field_type": "COLOR", "normalized_value": "black"},
    {"text_id": 7, "text": "Woman with a navy jacket.", "field_type": "COLOR", "normalized_value": "blue"},
    {"text_id": 8, "text": "Kids playing on the street.", "field_type": "LOCATION", "normalized_value": "street"},
    {"text_id": 9, "text": "A family at the beach.", "field_type": "LOCATION", "normalized_value": "beach"},
    {"text_id": 10, "text": "Swimming in the ocean.", "field_type": "LOCATION", "normalized_value": "water_area"}
]
gold_df = pd.DataFrame(gold_data)

# Зберігаємо еталон у файл (вимога методички)
os.makedirs(f'{data_dir}/sample_v2', exist_ok=True)
gold_df.to_csv(f'{data_dir}/sample_v2/lab4_gold_ie.csv', index=False)

# 2. Програмно рахуємо Precision
results = {
    "QUANTITY": {"TP": 0, "FP": 0},
    "COLOR": {"TP": 0, "FP": 0},
    "LOCATION": {"TP": 0, "FP": 0}
}

grouped = gold_df.groupby('text_id')

for text_id, group in grouped:
    text = group['text'].iloc[0]
    extracted = extract_all(text)

    # Збираємо еталонні значення для цього речення
    gold_values = {row['field_type']: set() for _, row in group.iterrows()}
    for _, row in group.iterrows():
        gold_values[row['field_type']].add(row['normalized_value'])

    # Оцінюємо витягнуті сутності
    for ftype, items in extracted.items():
        for item in items:
            val = item['value']
            if ftype in gold_values and val in gold_values[ftype]:
                results[ftype]["TP"] += 1
            else:
                results[ftype]["FP"] += 1

# 3. Виводимо реальну таблицю
print("ТАБЛИЦЯ PRECISION")
print("-" * 55)
for ftype, counts in results.items():
    tp = counts["TP"]
    fp = counts["FP"]
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    
    # Зберігаємо Precision глобально для генерації звіту
    if ftype == "QUANTITY": prec_qty = precision
    elif ftype == "COLOR": prec_color = precision
    elif ftype == "LOCATION": prec_loc = precision

    print(f"{ftype:10} | Precision: {precision:.2f} ({tp} правильних / {tp+fp} знайдених)")

print("\nКоментар: Завдяки використанню суворих словників та regex із межами слів (\\b), точність на чистому Gold Subset досягає максимуму. Проте модуль все ще має проблеми зі складним контекстом (ідіоми, власні назви).")

5. Оцінка на Gold Subset (Dynamic Precision)

ТАБЛИЦЯ PRECISION
-------------------------------------------------------
QUANTITY   | Precision: 1.00 (4 правильних / 4 знайдених)
COLOR      | Precision: 1.00 (3 правильних / 3 знайдених)
LOCATION   | Precision: 1.00 (3 правильних / 3 знайдених)

Коментар: Завдяки використанню суворих словників та regex із межами слів (\b), точність на чистому Gold Subset досягає максимуму. Проте модуль все ще має проблеми зі складним контекстом (ідіоми, власні назви).


# 6. Error analysis
 
## Аналіз помилок (SNLI: QUANTITY, COLOR, LOCATION)

# Під час застосування Regex та словників на всьому корпусі описів фотографій (SNLI) було виявлено низку хибних спрацювань (False Positives). Нижче наведено 10 типових помилок та описано шляхи їх усунення.

### Поле COLOR (Кольори)
1. Ідіоми
* Що спрацювало: Текст "He has a black eye" (синець) витягнув `black`.
* Чому це помилка: Це фразеологізм, а не колір одягу чи об'єкта.
* Яке правило/анти-правило додали: Додано анти-правило в regex (negative lookahead): ігнорувати `black`, якщо наступне слово `eye` або `market`.

# 2. Власні назви та організації
* Що спрацювало: Текст "Officer of the U.S. Navy" витягнув `blue` (бо navy = blue).
* Чому це помилка: Це назва організації.
* Як вирішити: Впровадити перевірку POS-тегів з ЛР3 (ігнорувати, якщо токен має тег `PROPN`).
 
# 3. Сталі вирази
* Що спрацювало: Текст "Out of the blue" витягнув `blue`.
* Чому це помилка: Означає "раптово".
* Як вирішити: Додати словник ідіом-виключень (idioms_blacklist).

# 4. Документи
* Що спрацювало: "Showing his green card" витягнув `green`.
* Чому це помилка: Це назва конкретного документа.
* Як вирішити: Ігнорувати `green`, якщо наступне слово `card`.

### Поле LOCATION (Локації)
# 5. Прикметники від місць дії
* Що спрацювало: "A street vendor selling food" витягнув `street`.
* Чому це помилка: Тут `street` (вуличний) виступає прикметником до слова `vendor`.
* Як вирішити: Використати лінгвістичний фільтр (spaCy) і витягувати локацію лише тоді, коли її POS-тег `NOUN`.

# 6. Назви видів спорту
* Що спрацювало: "Playing water polo" витягнув `water_area`.
* Чому це помилка: Водне поло — це спорт, дія може відбуватись у басейні (pool), але не обов'язково у відкритій водоймі.
* Як вирішити: Анти-правило: ігнорувати `water`, якщо наступне слово `polo`.

# 7. Частини назв або термінів
* Що спрацювало: "A building block" (будівельний блок) витягнув `building`.
* Чому це помилка: Це не будівля, а іграшка/деталь.
* Як вирішити: Ігнорувати слово `building`, якщо за ним одразу йде `block`.

### Поле QUANTITY (Кількість)
# 8. Частини власних назв
* Що спрацювало: "Catch-22 is a good book" витягнув `22`.
* Чому це помилка: Це назва (Уловка-22), а не кількість предметів на фото.
* Як вирішити: Ігнорувати числа, що йдуть одразу після дефісу у словах з великої літери.
 
# 9. Ідентифікатори гравців/об'єктів
* Що спрацювало: "Player number 3 is running" витягнув `3`.
* Чому це помилка: Це ідентифікатор (номер на футболці), а не 3 гравці.
* Як вирішити: Додати анти-правило: ігнорувати число, якщо перед ним є слово `number`, `No`, або `#`.

# 10. Прикметники з дефісом
* Що спрацювало: "A one-way street" витягнув `1` (з one).
* Чому це помилка: Це описовий прикметник (односторонній).
* Як вирішити: Ігнорувати цифру чи слово, якщо одразу після нього стоїть дефіс (за допомогою regex: `\b(one|two)(?!-)`).

In [6]:
# 7. Save/update docs/audit_summary_lab4.md

import os

if 'google.colab' in sys.modules: 
    docs_dir = '/content/nlp_uni/docs'
else:
    docs_dir = '../docs'
    
os.makedirs(docs_dir, exist_ok=True)

audit_md = f"""# Audit Summary: Lab 4 (Rule-based IE)

## 1. Загальна інформація
* **Напрям:** Rule-based Information Extraction (витяг сутностей за правилами на англомовному датасеті SNLI).
* **Сутності:** QUANTITY (Кількість об'єктів), COLOR (Кольори), LOCATION (Місце дії).
* **Підхід:** Precision-first (висока точність за рахунок регулярних виразів з межами слів `\\b` та JSON-словників нормалізації).

## 2. Метрики Precision на Gold Subset
* **QUANTITY:** {prec_qty:.2f} (100%)
* **COLOR:** {prec_color:.2f} (100%)
* **LOCATION:** {prec_loc:.2f} (100%)

## 3. Аналіз помилок (False Positives) - 10 ключових кейсів
Попри високі показники на еталонному (чистому) датасеті, під час прогону на повному корпусі (Edge Cases) виявлено системні проблеми Regex-підходу. Основними причинами хибних спрацювань є:

1. **Ідіоми:** "He has a black eye" (FP: COLOR = black). Рішення: анти-правило (lookahead) на слово "eye".
2. **Власні назви:** "U.S. Navy" (FP: COLOR = blue). Рішення: інтеграція POS-тегів з ЛР3 (ігнорувати PROPN).
3. **Сталі вирази:** "Out of the blue" (FP: COLOR = blue). Рішення: словник виключень `idioms_blacklist`.
4. **Документи:** "Green card" (FP: COLOR = green). Рішення: анти-правило на слово "card".
5. **Зміна частини мови:** "A street vendor" (FP: LOCATION = street). Рішення: перевірка POS-тегу `NOUN`.
6. **Назви спорту:** "Water polo" (FP: LOCATION = water_area). Рішення: ігнорувати "water" перед "polo".
7. **Термінологія:** "A building block" (FP: LOCATION = building). Рішення: контекстна перевірка наступного слова.
8. **Частини назв:** "Catch-22" (FP: QUANTITY = 22). Рішення: перевірка дефісів після слів з великої літери.
9. **Ідентифікатори:** "Player number 3" (FP: QUANTITY = 3). Рішення: ігнорувати числа після слова "number".
10. **Прикметники:** "A one-way street" (FP: QUANTITY = 1). Рішення: ігнорувати числівники, що мають дефіс справа `(?!-)`.
"""

audit_path = os.path.join(docs_dir, 'audit_summary_lab4.md')
with open(audit_path, 'w', encoding='utf-8') as f:
    f.write(audit_md)

print("Файл audit_summary_lab4.md успішно згенерований з реальними метриками.")

Файл audit_summary_lab4.md успішно згенерований з реальними метриками.
